# 案例：中心控制器编排与领域智能体协作

> Last updated: 2026-11-18

## 核心学习目标

本案例以多城市旅行规划为对象，学习如何设计领域智能体角色，并让中心编排器根据任务能力完成角色匹配、依赖排序和统一调度。

## 案例设计

## 中心编排流程

1. 中心智能体先判断请求类型。非旅行请求直接由大模型回复，不启动后续复杂流程。
2. 旅行请求先复用事实调查框架识别已知条件、待查信息和不确定项，再将目标拆解为子任务并分析依赖。
3. 独立的路由模块根据领域智能体职责、工具能力和输入输出定义，为每个子任务匹配最合适的智能体角色。
4. 中心控制器按依赖顺序调用领域智能体，全部任务完成后由最终规划智能体生成统一答复。

`fact_survey.py` 提供跨章节事实调查模板；`task_plan.py` 负责任务拆解和依赖校验；`agent_routing.py` 独立负责任务到智能体的匹配；`domain_agents.py` 定义领域角色和能力；`orchestrator.py` 用代码控制请求分诊、规划、路由和执行。

**学习重点**：
1. **领域智能体角色设定**：理解如何用职责描述、工具、输入输出和提示词定义一个领域智能体。
2. **任务与智能体角色匹配**：理解拆解后的子任务如何根据能力边界匹配到合适的领域角色。
3. **中心化编排**：理解中心编排器如何完成事实调查、任务拆解、依赖排序、执行调度和最终结果选择。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 模型访问 | 模型创建、文本调用、JSON 与结构化输出 | `llm.py` |
| 请求分诊 | 区分旅行任务与非旅行任务，非旅行任务直接回复 | `orchestrator.py` |
| 事实调查 | 四类事实分析与相对日期换算 | `fact_survey.py` |
| 任务规划 | 任务拆解、依赖分析和执行顺序校验 | `task_plan.py` |
| 角色匹配 | 子任务到领域智能体的能力路由 | `agent_routing.py` |
| 中心编排 | 统一执行、分层调度、最终回复选择与记忆写入 | `orchestrator.py` |
| 领域角色 | 领域智能体的职责、工具、输入输出和提示词 | `domain_agents.py` |
| 长期记忆 | 复用 Chapter-3 的短期上下文、Chroma 检索和记忆写入 | `memory_adapter.py` |
| 最终规划 | 汇总领域结果并选择最终用户答复 | `orchestrator.py` |

## 1. 环境与模块导入

本案例使用仓库根目录的共享 `.venv`。环境变量统一从仓库根目录 `.env` 加载。

In [13]:
import sys
from pathlib import Path

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

chapter6_dir = Path.cwd()
if not (chapter6_dir / "config.py").exists():
    chapter6_dir = Path.cwd() / "Chapter-6"
if not (chapter6_dir / "config.py").exists():
    raise FileNotFoundError("Could not locate Chapter-6/config.py")
if str(chapter6_dir) not in sys.path:
    sys.path.insert(0, str(chapter6_dir))

from config import BOOK_ROOT, CH6_DIR, CHROMA_DIR, load_project_dotenv

env_path = load_project_dotenv()
print("Chapter-6:", CH6_DIR)
print("项目根目录:", BOOK_ROOT)
print("环境文件:", env_path)
print("Chroma 目录:", CHROMA_DIR)

Chapter-6: E:\Book\agent-systems-book\Chapter-6
项目根目录: E:\Book\agent-systems-book
环境文件: E:\Book\agent-systems-book\.env
Chroma 目录: E:\Book\agent-systems-book\Chapter-6\chroma_memory


In [14]:
import importlib
import io
import json
import uuid
from contextlib import redirect_stdout

import agent_routing
import domain_agents
import fact_survey
import memory_adapter
import orchestrator
import task_plan

for module in (
    fact_survey,
    task_plan,
    agent_routing,
    domain_agents,
    memory_adapter,
    orchestrator,
):
    importlib.reload(module)

from domain_agents import DOMAIN_AGENT_SPECS
from orchestrator import CENTRAL_AGENT_SYSTEM_PROMPT, CentralOrchestrator

print("模块导入完成")

模块导入完成


## 2. 领域智能体角色设定

本章定义了 **6 个领域智能体**，分别承担旅行规划中的不同专业职责：

| 领域智能体 | 主要职责 |
|---|---|
| `WeatherAgent` | 根据城市和确切日期查询天气预报；每次请求只查询一个城市、一个日期的天气 |
| `AttractionAgent` | 根据城市和旅行偏好推荐景点；每次请求只检索一个城市的景点候选 |
| `HotelAgent` | 根据城市推荐酒店和住宿；每次请求只执行一次酒店推荐操作 |
| `RestaurantAgent` | 根据位置推荐餐厅；每次请求只查询一组餐厅推荐结果 |
| `FlightAgent` | 根据出发城市、到达城市和确切日期查询航班信息；每次请求只查询一组航线结果 |
| `ItineraryAgent` | 综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复 |

这些角色的详细定义集中在 `domain_agents.py`：`AgentSpec` 声明角色身份、职责描述、工具、输入输出和系统提示词，`ToolSpec` 声明可调用工具及其参数，`DomainAgentRegistry` 将定义汇总为后续任务规划和智能体路由使用的能力目录。

In [15]:
print("=== 领域智能体角色定义 ===\n")
for spec in DOMAIN_AGENT_SPECS.values():
    print(f"[{spec.name}] {spec.description}")
    for tool_spec in spec.tools:
        print(f"  工具: {tool_spec.name}")
        print(f"  输入: {tool_spec.parameters}")
        print(f"  输出: {tool_spec.returns}")
    if not spec.tools:
        print("  工具: 无（只负责最终规划与总结）")
        print(f"  输入: {spec.input_fields}")
        print(f"  输出: {spec.output_fields}")
    prompt_preview = " ".join(spec.system_prompt.split())
    print(f"  提示词: {prompt_preview[:140]}...")
    print()

=== 领域智能体角色定义 ===

[WeatherAgent] 根据城市和确切日期查询天气预报；每次请求只查询一个城市、一个日期的天气。
  工具: get_weather
  输入: ['city', 'date']
  输出: ['city', 'date', 'forecast', 'count', 'data_source', 'note']
  提示词: 你是天气查询领域智能体。 当前日期（本地时间）：2026-11-19 职责： 1. 使用可用天气工具完成当前天气子任务 2. 工具结果为空时，明确说明第 6 章尚未接入实时天气数据 3. 多日查询应基于工具实际返回的数据回答，不得补造缺失日期 4. 当前消息是明确天气子任务时，专...

[AttractionAgent] 根据城市和旅行偏好推荐景点；每次请求只检索一个城市的景点候选。
  工具: recommend_attractions
  输入: ['city', 'preferences', 'limit']
  输出: ['city', 'attractions', 'count', 'data_source']
  提示词: 你是景点推荐领域智能体。 职责： 1. 使用可用景点工具完成当前子任务 2. 只能从工具返回的景点候选中推荐，不得补充未返回的景点 3. 推荐理由只能基于名称、地址、区域、评分和类型等已有字段，不得编造简介、开放时间或门票 4. 非景点相关问题，回复：我只能协助景点推荐 注意： ...

[HotelAgent] 根据城市推荐酒店和住宿；每次请求只执行一次酒店推荐操作。
  工具: recommend_hotel
  输入: ['city', 'preferences', 'budget_cny_per_night_max']
  输出: ['city', 'hotels', 'count', 'data_source']
  提示词: 你是酒店推荐领域智能体。 规则： 1. 使用可用酒店工具完成当前子任务。 2. 安静、舒适等主观属性不由工具提供，不得声称已经核验。 3. 工具返回酒店候选后，只能基于区域、地址、参考价格、评分和类型推荐 3–5 家酒店。 每家需包含：名称、地址/位置、参考价格、评分（如有）、1...

[RestaurantAgent] 根据位置推

## 3. 用户输入

本案例使用一个多城市旅行规划问题。该请求同时包含出发地、目的地、时间、交通、住宿、天气和美食需求，适合作为后续任务拆解和依赖分析的输入。

In [16]:
user_query = """
我想带父母和孩子去北京玩3天2晚，预算8000元以内。
大概27年元旦假期，从南京出发，希望去故宫、长城和一个适合孩子的科技类场馆。
父母不适合长时间步行，每天步行最好不要超过8000步。
如果遇到下雨，希望自动调整为室内项目。请给出每天的行程安排、交通建议、预算估计和备选方案。
"""

print("用户输入：")
print(user_query.strip())

用户输入：
我想带父母和孩子去北京玩3天2晚，预算8000元以内。
大概27年元旦假期，从南京出发，希望去故宫、长城和一个适合孩子的科技类场馆。
父母不适合长时间步行，每天步行最好不要超过8000步。
如果遇到下雨，希望自动调整为室内项目。请给出每天的行程安排、交通建议、预算估计和备选方案。


## 4. 中心智能体 System Prompt 与请求分诊
中心 System Prompt 定义在 `orchestrator.py`，负责把请求分为两类：非旅行请求直接回答；旅行请求进入后续规划、路由和执行流程。

下方调用 `decide_request_route()` 执行请求分诊，并确认当前用户输入进入旅行任务链路。

In [17]:
from IPython.display import Markdown, display

display(Markdown("```text\n" + CENTRAL_AGENT_SYSTEM_PROMPT + "\n```"))

ORCHESTRATION_THREAD_ID = "central_orchestration_entry_001"
orchestrator_instance = CentralOrchestrator(enable_memory=True)
request_route = await orchestrator_instance.decide_request_route(user_query)

assert request_route["route"] == "travel"
print(f"请求类型: {request_route['route']}")

```text
你是旅行规划应用的中心编排智能体，负责请求分诊和后续流程控制。

## 任务类型判断
将用户请求分为两类：

1. 旅行任务
包括旅行行程规划，以及天气、景点、酒店、餐厅、航班、交通等旅行相关信息查询。这类任务需要后续的事实调查、任务规划、领域智能体分配和执行，不能直接凭模型知识作答。

2. 非旅行任务
包括问候、写作、编程、常识问答、数学计算以及与旅行无关的信息查询。这类任务不需要启动任务规划或多智能体执行。

## 处理规则
1. 如果是非旅行任务：直接使用你自己的通用能力回答；


2. 如果是旅行任务：
   - 不要直接生成最终答案；
   - 将任务交给后续流程：先完成事实调查、任务拆解和依赖分析，再匹配领域智能体，最后按执行计划调用领域智能体；
   - response 字段留空，等待后续流程生成最终答案。

## 输出格式
只输出 JSON 对象，不要输出解释文字：

非旅行任务：
{"route": "direct", "response": "直接给用户的完整回答"}

旅行任务：
{"route": "travel", "response": ""}

```

✓ 长期记忆系统已启用（Chapter-3）
请求类型: travel


## 5. 事实调查与长期记忆检索

先调用 `task_plan.survey_facts()`，区分用户已给出的事实、待查信息、待推导信息和不确定项；随后通过 `MemoryAdapter` 复用 Chapter 3 的长期记忆检索能力，查找与当前请求相关的用户偏好。

事实调查结果和长期记忆会作为后续任务拆解的背景信息，不直接作为领域事实答案。

In [18]:
pre_survey = await task_plan.survey_facts(
    orchestrator_instance.llm,
    user_query,
)
print("=== 事实调查结果 ===\n")
print(json.dumps(
    {key: value for key, value in pre_survey.items() if key != "raw_text"},
    ensure_ascii=False,
    indent=2,
))

memories = []
if orchestrator_instance.memory_adapter:
    hits = await orchestrator_instance.memory_adapter.search_memories_async(user_query)
    memories = orchestrator_instance.memory_adapter.format_memories_for_plan(hits)
print(f"\n检索到的长期记忆: {len(memories)} 条")

=== 事实调查结果 ===

{
  "given_facts": [
    "出发地：南京",
    "目的地：北京",
    "出行时间：2027年元旦假期（相对日期，需换算为具体日期）",
    "行程时长：3天2晚",
    "总预算：8000元以内",
    "同行人员：父母、孩子（暗示为老中青三代同游）",
    "必去景点：故宫、长城、一个适合孩子的科技类场馆",
    "健康约束：父母不适合长时间步行，每日步行不超过8000步",
    "天气应对：若遇下雨，自动调整为室内项目"
  ],
  "facts_to_lookup": [
    "2027年元旦假期的具体放假日期及天数（需确认是否包含调休）",
    "2026-11至2027-01期间南京到北京的高铁/飞机票价及时刻表",
    "北京当地住宿在2027年元旦期间的价格区间",
    "故宫、长城（如八达岭或慕田峪）、科技类场馆（如中国科学技术馆）的门票价格及预约政策",
    "各景点内部的无障碍设施情况及省力游览路线（如缆车、电瓶车服务）",
    "北京2027年1月初的历史天气数据及降水概率",
    "北京市内适合老人的交通方式（如包车、出租车）费用",
    "备选室内科技类或文化类场馆列表（用于雨天替换）"
  ],
  "facts_to_derive": [
    "将“2027年元旦假期”转换为具体的YYYY-MM-DD日期范围",
    "根据8000元总预算，扣除交通和住宿后，计算每日餐饮、门票及市内交通的可支配额度",
    "规划每日行程路线，确保单日步行步数控制在8000步以内（需结合地图距离和游览强度估算）",
    "制定雨天备选方案，将原定的室外景点（如长城部分路段）替换为室内活动",
    "评估不同长城段落的体力消耗，选择最适合老人且符合步数限制的段落"
  ],
  "educated_guesses": [
    "“孩子”可能指学龄前或小学阶段儿童，对互动性强的科技场馆感兴趣",
    "2027年元旦期间北京气温较低，可能在0度以下，需考虑保暖因素",
    "故宫和热门科技场馆在节假日期间票源紧张，需提前预约",
    "考虑到父母体力限制，长城行程可能倾向于选择有缆车直达且坡度较缓

## 6. 任务拆解与依赖分析

任务拆解、依赖识别和拓扑排序已在 Chapter 4 的任务分解与规划案例中详细学习，本章不再分别展开这两部分。

这里把任务拆解和依赖分析合并为一个代码单元格，直接调用 `task_plan.create_task_plan()`，一次得到业务子任务、任务依赖和执行顺序。

In [19]:
task_plan_result = await task_plan.create_task_plan(
    orchestrator_instance.llm,
    user_query,
    pre_survey,
    memories,
    orchestrator_instance.agent_registry,
)

print("=== 任务拆解与依赖分析结果 ===\n")
for task in task_plan_result["tasks"]:
    print(f"[{task['task_id']}] {task['description']}")
    print(f"  依赖: {task.get('depends_on', [])}")

print("\n执行顺序:", " → ".join(task_plan_result["execution_order"]))

=== 任务拆解与依赖分析结果 ===

[T1] 根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-01的天气
  依赖: []
[T2] 根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-02的天气
  依赖: []
[T3] 根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-03的天气
  依赖: []
[T4] 根据出发城市、到达城市和确切日期查询航班信息：查询出发城市南京，到达城市北京，日期2026-12-31的航班信息
  依赖: []
[T5] 根据出发城市、到达城市和确切日期查询航班信息：查询出发城市北京，到达城市南京，日期2027-01-03的航班信息
  依赖: []
[T6] 根据城市推荐酒店和住宿：推荐城市北京的酒店和住宿
  依赖: []
[T7] 根据城市和旅行偏好推荐景点：检索城市北京的景点候选，偏好包含故宫、长城及适合孩子的科技类场馆
  依赖: []
[T8] 根据位置推荐餐厅：查询一组北京市内的餐厅推荐结果
  依赖: []
[T9] 综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复：综合上述查询结果，结合总预算8000元以内、每日步行不超过8000步的健康约束以及下雨自动调整为室内项目的应急偏好，生成包含每天行程安排、交通建议、预算估计和备选方案的最终答复
  依赖: ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8']

执行顺序: T1 → T2 → T3 → T4 → T5 → T6 → T7 → T8 → T9


## 7. 智能体路由提示词

任务规划完成后，`agent_routing.py` 负责为每个业务子任务选择领域智能体，并从事实调查结果中提取该智能体工具需要的参数。

路由提示词接收四类输入：可用领域智能体及技能、事实调查结果、待分配任务、已验证的依赖关系。模型内部输出结构化路由记录，代码随后校验智能体名称、过滤无效参数，并保留任务原文和依赖关系。

In [20]:
from agent_routing import AGENT_ROUTING_PROMPT

display(Markdown("### 智能体路由提示词\n\n```text\n" + AGENT_ROUTING_PROMPT + "\n```"))

### 智能体路由提示词

```text
你是中心智能体的「子任务路由模块」。根据已拆解的子任务，为每个子任务选择最合适的领域智能体并提取调用参数。

## 可用领域智能体
{agent_team}

## 事实调查结果
{pre_survey_json}

## 子任务列表
{subtasks_json}

## 依赖分析结果
以下 depends_on 已通过本地校验，必须原样使用：
{depends_on_json}

## 输出要求
只输出 JSON 数组，其中每个元素对应一个子任务：
[
  {{
    "task_id": "T1",
    "agent": "WeatherAgent",
    "params": {{"city": "上海", "date": "2026-05-01"}}
  }}
]

规则：
1. agent 必须是可用领域智能体之一。
2. 每个子任务只分配一个 agent。
3. params 字段名必须匹配该 agent 的 inputSchema。
4. 必须为每个 task_id 输出一条记录。
5. 不要输出 description、depends_on 或其他解释文字，任务原文和依赖关系由本地代码保留。

```

## 8. 智能体路由执行与结果

调用独立的 `agent_routing.route_to_agents()`，把每个业务子任务分配给最合适的领域智能体。路由结果保留任务原文、验证后的依赖关系和提取出的工具参数；智能体名称和参数字段都会经过本地校验。

下发给领域智能体的是自然语言子任务，而不是路由模块内部的 JSON。下面展示本章最关心的问题：“子任务 → 领域智能体”的匹配结果。

In [21]:
depends_map = {
    task["task_id"]: list(task.get("depends_on") or [])
    for task in task_plan_result["tasks"]
}
validated_order = task_plan_result["execution_order"]
assigned_tasks = await orchestrator_instance.agent_routing.route_to_agents(
    task_plan_result["tasks"],
    validated_order,
    depends_map,
    pre_survey,
)
execution_plan = {
    "pre_survey": task_plan_result["pre_survey"],
    "pre_survey_raw": task_plan_result["pre_survey_raw"],
    "retrieved_memories": task_plan_result["retrieved_memories"],
    "total_goal": task_plan_result["total_goal"],
    "plan_fallback": task_plan_result["plan_fallback"],
    "subtasks": assigned_tasks,
    "execution_order": validated_order,
}

print("=== 智能体路由结果 ===\n")
for task in assigned_tasks:
    print(f"[{task['task_id']}] {task['description']}")
    print(f"  → {task.get('agent') or '未分配'}")
    if task.get("assignment_error"):
        print(f"  路由错误: {task['assignment_error']}")
    print()

=== 智能体路由结果 ===

[T1] 根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-01的天气
  → WeatherAgent

[T2] 根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-02的天气
  → WeatherAgent

[T3] 根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-03的天气
  → WeatherAgent

[T4] 根据出发城市、到达城市和确切日期查询航班信息：查询出发城市南京，到达城市北京，日期2026-12-31的航班信息
  → FlightAgent

[T5] 根据出发城市、到达城市和确切日期查询航班信息：查询出发城市北京，到达城市南京，日期2027-01-03的航班信息
  → FlightAgent

[T6] 根据城市推荐酒店和住宿：推荐城市北京的酒店和住宿
  → HotelAgent

[T7] 根据城市和旅行偏好推荐景点：检索城市北京的景点候选，偏好包含故宫、长城及适合孩子的科技类场馆
  → AttractionAgent

[T8] 根据位置推荐餐厅：查询一组北京市内的餐厅推荐结果
  → RestaurantAgent

[T9] 综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复：综合上述查询结果，结合总预算8000元以内、每日步行不超过8000步的健康约束以及下雨自动调整为室内项目的应急偏好，生成包含每天行程安排、交通建议、预算估计和备选方案的最终答复
  → ItineraryAgent



## 9. 领域智能体执行结果

调用 `orchestrator.execute_plan()`，按依赖层顺序执行领域智能体；同一层中，仅声明支持并发的智能体并行执行。

存在依赖时，当前领域智能体会收到前序子任务、前序任务下发的领域智能体以及该智能体的回复。工具原始结果不继续向下游传递。`message_records` 用于保存并展示这些实际下发内容。

In [22]:
run_tag = uuid.uuid4().hex[:8]
agent_messages = []
execution_log = io.StringIO()
with redirect_stdout(execution_log):
    subtask_results = await orchestrator_instance.execute_plan(
        execution_plan,
        ORCHESTRATION_THREAD_ID,
        run_tag,
        message_records=agent_messages,
    )

print("=== 带依赖关系的领域智能体实际输入 ===\n")
for record in agent_messages:
    message = record["message"]
    if "前序依赖结果" not in message:
        continue
    print(f"[{record['task_id']}] → {record['agent']}")
    print(message)
    print()


print("=== 领域智能体执行结果 ===\n")
for task_id in execution_plan["execution_order"]:
    task_result = subtask_results.get(task_id, {})
    tool_data = task_result.get("tool_data")
    fields = list(tool_data) if isinstance(tool_data, dict) else []
    summary = " ".join((task_result.get("agent_summary") or "").split())
    print(f"[{task_id}] {task_result.get('agent', '?')}")
    print(f"  智能体输出: {summary[:240]}{'…' if len(summary) > 240 else ''}")
    print()

=== 带依赖关系的领域智能体实际输入 ===

[T9] → ItineraryAgent
【子任务 T9 · ItineraryAgent】
请完成下方子任务，并在调用工具后用中文总结结果。
即使任务说明很长或提到多城市/其他模块，也只完成本子任务；不要回复「我只能协助…」拒答。
子任务：综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复：综合上述查询结果，结合总预算8000元以内、每日步行不超过8000步的健康约束以及下雨自动调整为室内项目的应急偏好，生成包含每天行程安排、交通建议、预算估计和备选方案的最终答复
工具参数：cities=['南京', '北京']，dates=['2026-12-31', '2027-01-01', '2027-01-02', '2027-01-03']，preferences=总预算8000元以内，每日步行不超过8000步，下雨自动调整为室内项目，老中青三代同游，departure_city=南京，destination_city=北京，days=3
前序依赖结果（作为本任务输入）：
- 前序任务 T1：根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-01的天气
  下发给智能体：WeatherAgent
  智能体回复：根据工具返回结果，第 6 章尚未接入实时天气数据，因此无法提供北京在 2027-01-01 的具体天气预报信息。后续章节会补充相关数据源。
- 前序任务 T2：根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-02的天气
  下发给智能体：WeatherAgent
  智能体回复：查询结果：北京在 2027-01-02 的天气数据为空。

说明：第 6 章尚未接入实时天气数据，因此无法提供具体的气温、降水等预报信息。后续章节会补充相关数据源。
- 前序任务 T3：根据城市和确切日期查询天气预报：查询城市北京，日期2027-01-03的天气
  下发给智能体：WeatherAgent
  智能体回复：查询结果：北京 2027-01-03 的天气预报数据为空。

说明：第 6 章尚未接入实时天气数据，后续章节会补充具体天气来源。当前无法提供该日期的气温、降水等详细天气信息。
- 前序任务 T4：根据出发城市、到达城市和确切日期查询航班信息：查询

## 10. 最终旅行规划

调用 `compose_final_response()`，选择最后生成的旅行规划结果作为最终答复。`ItineraryAgent` 不调用领域工具，只综合依赖任务返回的天气、景点、酒店、餐厅、航班和用户偏好生成日程。

In [23]:
final_log = io.StringIO()
with redirect_stdout(final_log):
    final_response = orchestrator_instance.compose_final_response(
        subtask_results,
        execution_plan["execution_order"],
    )

if orchestrator_instance.memory_adapter:
    orchestrator_instance.memory_adapter.record_turn(
        ORCHESTRATION_THREAD_ID,
        user_query,
        final_response,
    )
    await orchestrator_instance.memory_adapter.ingest(
        f"用户请求: {user_query.strip()}\n偏好摘要: {final_response[:500]}",
        memory_type="preference",
    )

display(Markdown("### 最终旅行规划\n\n" + final_response))

### 最终旅行规划

您好，我是您的行程规划助手。根据您提供的依赖任务查询结果，我为您整理了从南京出发前往北京的旅行规划建议。

**重要提示：数据缺失说明**
在生成最终日程前，必须向您如实说明当前系统数据的局限性，这将直接影响行程的精确度：
1.  **航班信息缺失**：系统未接入实时航班数据，无法提供2026-12-31（去程）和2027-01-03（返程）的具体航班号、时刻及价格。您需要自行通过航空公司或OTA平台预订往返机票/高铁。
2.  **天气信息缺失**：系统无法提供北京2027-01-01至01-03的具体天气预报。鉴于北京冬季寒冷且可能降雪，建议您出发前查看实时天气，并默认按“低温/可能雨雪”准备衣物。
3.  **景点信息缺失**：系统未检索到有效的景点推荐结果（包括您偏好的故宫、长城及科技馆）。因此，以下行程中的景点部分仅基于北京常规旅游常识进行逻辑占位，**具体门票预约、开放时间及内部路线需您另行核实**。

尽管存在上述数据缺口，结合已获取的**酒店**和**餐厅**数据，以及您的**预算（8000元以内）**、**健康约束（每日步行<8000步）**和**老中青三代同游**偏好，我为您制定了以下以“低体力消耗、室内优先、家庭友好”为核心的参考日程。

---

### 📅 最终旅行日程建议 (2026-12-31 至 2027-01-03)

#### **第一天：2026-12-31（周三） | 南京 ✈️ 北京 | 抵达与休整**
*   **交通**：请自行预订南京至北京的早班机或高铁，建议中午前抵达，以减少老人旅途疲劳。
*   **住宿建议**：入住 **汉庭酒店(北京同仁医院崇文门地铁站店)**
    *   *理由*：位于东城区核心，紧邻地铁，交通便利，方便后续行程，评分4.7，适合家庭。
*   **下午安排**：酒店办理入住后，建议在酒店附近休息，避免长途跋涉。若精力允许，可步行至附近的**崇文门商圈**简单逛逛（室内商场，避风保暖）。
*   **晚餐推荐**：**荣小馆(金融街购物中心店)**
    *   *注意*：该店位于西城区金融街，距离崇文门有一定距离，建议打车前往。
    *   *特色*：环境优越，评分5.0，人均148元，适合跨年夜的家庭聚餐，菜品精致适合老少口味。
*   **步行预估**：< 3000步（主要为酒店内及短途移动）。

#### **第二天：2027-01-01（周四） | 北京 | 文化体验（室内为主）**
*   **上午安排**：由于景点数据缺失，且考虑冬季户外寒冷及老人体力，建议安排**室内文化场馆**（如国家博物馆或中国科学技术馆，需您自行确认预约）。此类场所地面平整，有电梯，符合“步行少、室内”的偏好。
*   **午餐推荐**：**潇湘府·家里饭(朝阳路·双桥店)** 或就近选择商场内餐厅。
    *   *特色*：人均90元，评分5.0，“家里饭”氛围浓厚，性价比高，适合家庭用餐。
*   **下午安排**：继续室内活动或返回酒店午休。避免前往长城等需要大量爬山的户外景点，以遵守“每日步行不超过8000步”的健康约束。
*   **晚餐推荐**：**德仁居烤鸭店(管庄店)**
    *   *特色*：人均70元，评分4.7，提供北京特色烤鸭，经济实惠，老人孩子通常喜爱。
*   **步行预估**：< 6000步（视具体室内场馆大小而定，建议多利用馆内休息区）。

#### **第三天：2027-01-02（周五） | 北京 | 休闲漫步与特色体验**
*   **上午安排**：建议前往 **天坛公园**（仅逛祈年殿周边平坦区域，不爬高）。
    *   *理由*：靠近之前推荐的喆啡酒店区域，公园开阔，空气好，适合老人散步，但需控制时长和步数。
*   **午餐推荐**：**阳坊胜利涮羊肉(阳坊老店)**
    *   *注意*：位于昌平区，距离市区较远。若选择此店，需预留充足交通时间。若觉得太远，可替换为市区的 **净悟真私宅烤全羊(机场旗舰店)**（朝阳区，人均128元，评分4.9，环境独特）。
    *   *建议*：考虑到带老人和孩子，**净悟真**可能更便于控制行程节奏。
*   **下午安排**：返回酒店休息，或在北京站内/附近购买特产。避免剧烈运动。
*   **晚餐**：建议在酒店附近简单用餐，或尝试酒店周边未列出的清淡餐饮，为次日返程储备体力。
*   **步行预估**：< 8000步（严格控制公园内游览范围）。

#### **第四天：2027-01-03（周六） | 北京 ✈️ 南京 | 返程**
*   **上午安排**：睡到自然醒，整理行李。检查是否有遗漏物品。
*   **午餐**：酒店附近简餐。
*   **交通**：请自行预订北京至南京的下午或傍晚航班/高铁。建议提前2小时到达机场/车站。
*   **住宿**：无（当晚返回南京）。

---

### 💰 预算估算 (总预算 8000元以内)

由于航班和景点门票数据缺失，以下仅为基于已知数据的估算，**实际支出需加上大交通和门票费用**：

1.  **住宿 (3晚)**：
    *   参考汉庭/喆啡等中端酒店，假设每晚400-600元（家庭房或两间房），3晚约 **1200 - 1800元**。
2.  **餐饮 (3天 x 3餐 + 零食)**：
    *   早餐：酒店含或路边店，约30元/人/天。
    *   正餐：参考推荐餐厅人均70-150元。假设4人（老中青三代），每顿正餐约400-600元。
    *   3天餐饮总计约 **1500 - 2000元**。
3.  **市内交通**：
    *   打车/地铁为主，预计 **300 - 500元**。
4.  **剩余预算用于大交通与门票**：
    *   目前已知支出约 3000 - 4300元。
    *   剩余 **3700 - 5000元** 可用于支付南京-北京往返机票/高铁票（4人）及景点门票。
    *   *注意*：若机票价格较高，可能超出8000元总预算，建议优先选择高铁或提前抢购折扣机票。

---

### ⚠️ 备选方案与应急策略

1.  **下雨/雪/恶劣天气应急**：
    *   由于无法获取实时天气，**默认所有户外行程（如天坛）均有室内备选**。
    *   若遇雨雪，取消天坛行程，改为参观 **中国国家博物馆** 或 **北京规划展览馆**（均为室内，有电梯，免风吹雨淋）。
    *   餐饮均选择在商场内或独立门店，避免露天排队。

2.  **体力不足应急**：
    *   若老人感到疲劳，立即取消当日下午行程，返回酒店休息。
    *   出行全程建议**打车**而非地铁换乘，以减少步行和站立时间，虽然成本略增，但符合“健康约束”优先级。

3.  **景点预约提醒**：
    *   北京热门景点（如故宫、国博）通常需提前7天预约。由于系统未提供景点数据，请您务必**立即**通过官方渠道查询并预约2027-01-01至01-02的门票，以免无法入园。

希望这份基于现有数据的规划能为您提供参考。祝您和家人旅途愉快，平安顺遂！

## 11. 非旅行请求直答

中心智能体识别出非旅行任务后，直接调用大模型回答，不进入事实调查、任务拆解、依赖分析和多智能体执行。

In [24]:
direct_query = "请用一句话解释什么是二分查找。"
with redirect_stdout(io.StringIO()):
    direct_result = await orchestrator_instance.process_request(
        direct_query,
        thread_id="direct_query_entry_001",
    )

assert direct_result["route"] == "direct"
assert direct_result["execution_plan"] is None
print(direct_result["final_response"])

二分查找是一种在有序数组中查找特定元素的算法，通过不断将搜索区间减半来快速定位目标值。


## 12. 小结

- 领域智能体通过职责描述、工具、输入输出和提示词统一定义。
- 中心智能体负责请求分诊，后续规划、路由和执行顺序由代码控制。
- `task_plan.py` 复用 Chapter 2 和 Chapter 4 的能力，负责事实调查、任务拆解和依赖分析。
- `agent_routing.py` 独立完成任务到领域智能体的匹配，路由结构不直接暴露给领域智能体。
- 存在依赖时，领域智能体接收前序任务、前序智能体和前序回复，不接收工具原始结果。
- `ItineraryAgent` 综合领域执行结果，生成完整旅行答复。